# RG3 RF/ExtraTrees 중첩 결합 선택
테스트정답 사용 없이 학습내부 F1로 25가지 결합/기준 선택.

In [1]:
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
s=''.join(json.loads((root/'notebooks/32_rg3_small_confirm_ensembles.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('records=[]')[0])
def stats(yy,p):
    tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
    return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
records=[];chosen_rows=[]
weights=[0.,.25,.5,.75,1.]
rates=[.0025,.005,.01,.015,.02]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        out=np.column_stack([model_pair(name,ref,q).max(axis=1) for name in ['RandomForest','ExtraTrees']])
        inside=np.full((len(ref),2),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():
                for k,name in enumerate(['RandomForest','ExtraTrees']):inside[mask,k]=model_pair(name,ref[~mask],ref[mask]).max(axis=1)
        valid=np.isfinite(inside).all(axis=1);yy=y[ref][valid];cal=inside[valid]
        choices=[]
        for w in weights:
            score=cal[:,0]*w+cal[:,1]*(1-w)
            for rate in rates:
                cut=float(np.quantile(score[yy==0],1-rate,method='higher'))
                tp,fp,fn,tn,f1=stats(yy,score>cut)
                choices.append(dict(weight_RF=w,rate=rate,threshold=cut,inner_F1=f1,inner_FP=fp))
        # Fixed conservative fallback if inner data contain no useful positive signal.
        selected=max(choices,key=lambda c:(c['inner_F1'],-c['inner_FP'],-abs(c['weight_RF']-1),-abs(c['rate']-.01)))
        if selected['inner_F1']==0:
            selected=next(c for c in choices if c['weight_RF']==1 and c['rate']==.01)
        for name,choice in [('selected_RF_ET',selected),('RF_baseline',next(c for c in choices if c['weight_RF']==1 and c['rate']==.01))]:
            score=out[:,0]*choice['weight_RF']+out[:,1]*(1-choice['weight_RF'])
            chosen_rows.append(dict(scheme=scheme,block=block,model=name,**choice))
            for i,z in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[i]),prediction=int(score[i]>choice['threshold'])))
        print('Completed nested blend',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tp,fp,fn,tn,f1=stats(g.actual,g.prediction)
    rows.append(dict(scheme=scheme,model=name,TP=tp,FP=fp,FN=fn,TN=tn,F1=f1,AP=average_precision_score(g.actual,g.score)))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_nested_blend_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_nested_blend_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(chosen_rows).to_csv(tables/'rg3_nested_blend_choices.csv',index=False,encoding='utf-8-sig')
lines=['# RG3 RF/ExtraTrees 학습 내부 결합 선택','',
'각 알고리즘은 기존 흔들림4/관계잔차4 두 분류기의 최대점수다. RF비율0/25/50/75/100%, 학습정상상위0.25/0.5/1/1.5/2% 경계25조합을 학습내부구간제외 F1로만 선택한다. 내부F1모두0이면기존RF/1%로돌아간다. 외부테스트정답을가중치/기준선택에사용하지않는다.',
'기존관측구간제외583샷/가스17, 과거학습440샷/가스13. 후보군선택은이전동일자료결과를보고정해최종독립평가가아니다. 내부정상기준은새구간오탐률을보장하지않는다. 원본센서이력가용조건/실제Run이아닌관측구간조건유지.',
'','| 평가 | 방식 | TP | FP | FN | TN | F1 | AP |','|---|---|---:|---:|---:|---:|---:|---:|']
for r in metrics.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} | {r.AP:.4f} |')
(root/'reports/rg3_nested_blend_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(metrics.to_string(index=False))


             scheme          model  TP  FP  FN  TN       F1       AP
leave_one_block_out selected_RF_ET   3   9  14 557 0.206897 0.116850
leave_one_block_out    RF_baseline   4   6  13 560 0.296296 0.168625
   past_blocks_only selected_RF_ET   1   7  12 420 0.095238 0.065281
   past_blocks_only    RF_baseline   2   4  11 423 0.210526 0.081203